# ADX + Directional Filter on SPY
## Strategy Brief
This strategy uses the Average Directional Index (ADX) combined with a directional filter to identify potential trend-following opportunities in SPY. The ADX is used to measure the strength of a trend, while the directional filter helps determine the trade direction. When the ADX indicates a strong trend and the directional filter aligns, a trade is initiated. The strategy aims to capture significant price movements by entering trades in the direction of the prevailing trend, potentially leading to favorable risk-adjusted returns.
## References
- (No external references)

In [ ]:
!pip install yfinance pandas numpy matplotlib scipy

### PHASE 1 - Trading Context
In this phase, we define the parameters for the ADX + Directional Filter strategy. These parameters will guide the computation of the indicators and the trading logic.

In [ ]:
ADX_PERIOD = 14
ADX_THRESHOLD = 25
START_DATE = '2010-01-01'
END_DATE = '2023-10-01'
SYMBOL = 'SPY'

### PHASE 2 - Data Exploration
We will download historical price data for SPY using yfinance, compute the ADX and directional filter, and visualize these indicators overlaid on the price chart.

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.signal import argrelextrema

# Download SPY data
data = yf.download(SYMBOL, start=START_DATE, end=END_DATE)

# Calculate ADX
high = data['High']
low = data['Low']
close = data['Close']

plus_dm = high.diff()
minus_dm = low.diff()

plus_dm[plus_dm < 0] = 0
minus_dm[minus_dm > 0] = 0

tr = np.maximum(high - low, np.maximum(abs(high - close.shift()), abs(low - close.shift())))
atr = tr.rolling(window=ADX_PERIOD).mean()

plus_di = 100 * (plus_dm.ewm(alpha=1/ADX_PERIOD).mean() / atr)
minus_di = abs(100 * (minus_dm.ewm(alpha=1/ADX_PERIOD).mean() / atr))

adx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di)).ewm(alpha=1/ADX_PERIOD).mean()

# Plot
plt.figure(figsize=(14, 7))
plt.plot(data['Close'], label='SPY Close')
plt.plot(adx, label='ADX')
plt.axhline(y=ADX_THRESHOLD, color='r', linestyle='--', label='ADX Threshold')
plt.title('SPY Price with ADX')
plt.legend()
plt.show()

### PHASE 3 - Strategy Engineering
We will create the trading signals based on the ADX and a simple directional filter. The directional filter will be based on the moving average crossover.

In [ ]:
# Directional Filter: Moving Average Crossover
short_ma = data['Close'].rolling(window=10).mean()
long_ma = data['Close'].rolling(window=30).mean()

directional_filter = np.where(short_ma > long_ma, 1, -1)

# Signal: ADX + Directional Filter
signal = np.where((adx > ADX_THRESHOLD) & (directional_filter == 1), 1, 0)
signal = np.where((adx > ADX_THRESHOLD) & (directional_filter == -1), -1, signal)

# Positions
positions = signal.shift(1).fillna(0)

### PHASE 4 - Coding & Backtesting
We will backtest the strategy by calculating daily returns based on the positions and plot the resulting equity curve.

In [ ]:
# Calculate daily returns
daily_returns = data['Close'].pct_change()
strategy_returns = daily_returns * positions

# Equity curve
equity_curve = (1 + strategy_returns).cumprod()

# Plot equity curve
plt.figure(figsize=(14, 7))
plt.plot(equity_curve, label='Strategy Equity Curve')
plt.plot((1 + daily_returns).cumprod(), label='Buy and Hold Equity Curve')
plt.title('Equity Curve')
plt.legend()
plt.show()

### PHASE 5 - Performance Evaluation
We will evaluate the strategy's performance using key metrics such as CAGR, Sharpe ratio, Sortino ratio, Calmar ratio, and maximum drawdown, and compare it to a buy-and-hold strategy.

In [ ]:
# Performance metrics
def calculate_cagr(equity_curve):
    n = len(equity_curve) / 252
    return (equity_curve[-1] / equity_curve[0]) ** (1/n) - 1

cagr = calculate_cagr(equity_curve)

# Sharpe Ratio
sharpe_ratio = strategy_returns.mean() / strategy_returns.std() * np.sqrt(252)

# Sortino Ratio
downside_returns = strategy_returns[strategy_returns < 0]
sortino_ratio = strategy_returns.mean() / downside_returns.std() * np.sqrt(252)

# Maximum Drawdown
roll_max = equity_curve.cummax()
drawdown = (equity_curve - roll_max) / roll_max
max_drawdown = drawdown.min()

# Calmar Ratio
calmar_ratio = cagr / abs(max_drawdown)

# Buy and Hold
buy_and_hold_cagr = calculate_cagr((1 + daily_returns).cumprod())

# Comparison table
performance_table = pd.DataFrame({
    'Metric': ['CAGR', 'Sharpe Ratio', 'Sortino Ratio', 'Calmar Ratio', 'Max Drawdown'],
    'Strategy': [cagr, sharpe_ratio, sortino_ratio, calmar_ratio, max_drawdown],
    'Buy and Hold': [buy_and_hold_cagr, np.nan, np.nan, np.nan, np.nan]
})

print(performance_table)

### PHASE 6 - Deploy & Monitor
We will create a function to download the last 60 days of SPY data, compute today's signal, and print the current position.

In [ ]:
def get_current_position():
    recent_data = yf.download(SYMBOL, period='60d')
    
    # Recalculate indicators for the recent data
    high = recent_data['High']
    low = recent_data['Low']
    close = recent_data['Close']
    
    plus_dm = high.diff()
    minus_dm = low.diff()
    
    plus_dm[plus_dm < 0] = 0
    minus_dm[minus_dm > 0] = 0
    
    tr = np.maximum(high - low, np.maximum(abs(high - close.shift()), abs(low - close.shift())))
    atr = tr.rolling(window=ADX_PERIOD).mean()
    
    plus_di = 100 * (plus_dm.ewm(alpha=1/ADX_PERIOD).mean() / atr)
    minus_di = abs(100 * (minus_dm.ewm(alpha=1/ADX_PERIOD).mean() / atr))
    
    adx = 100 * (abs(plus_di - minus_di) / (plus_di + minus_di)).ewm(alpha=1/ADX_PERIOD).mean()
    
    short_ma = close.rolling(window=10).mean()
    long_ma = close.rolling(window=30).mean()
    
    directional_filter = np.where(short_ma > long_ma, 1, -1)
    
    signal = np.where((adx > ADX_THRESHOLD) & (directional_filter == 1), 1, 0)
    signal = np.where((adx > ADX_THRESHOLD) & (directional_filter == -1), -1, signal)
    
    current_position = signal[-1]
    print(f"Current Position: {'Long' if current_position == 1 else 'Short' if current_position == -1 else 'Neutral'}")

get_current_position()